# Stages GPT-Image-1.5 midpoints, scores the 120-pair subset (`tab:generative` GPT rows)

Cell 2 stages GPT-Image-1.5 outputs into `tm_scale1000/{gpt,gpt_p1}`; cell 6/7 (a later, unsaved variant per docs/INVENTORY.md) produced `GPlaceMedgpt`, `GgptDRDin*`, and the `fig:gptstrip`/`fig:app-gpt` figure panels -- the code that wrote those saved values is not preserved in this copy, only the printed output.

**Needs:** the DTD 120-pair subset; GPT-Image-1.5 outputs (see `evidence/gpt_image_1_5_experiment/` for the actual prompts, call ledger, and generated images behind this -- proprietary API, not reproducible without an Azure OpenAI key).

**Maintained runnable path:** `experiments/generative_120.py` scores GPT-Image-1.5 outputs you already have (`--gpt-dir`/`--gptsym-dir`) the same way this notebook did. The staging/prompting step itself has no maintained script -- see `evidence/gpt_image_1_5_experiment/` instead.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import os, glob, json
import numpy as np, pandas as pd
BASE=os.environ.get('DATA_ROOT', '/content/data')  # was hardcoded to the Drive mount; set DATA_ROOT to point at your local copy
SCALE=f'{BASE}/tm_scale1000'; OUT=f'{BASE}/matching_ablation'

In [ ]:
# ============================================================
# GPT A. DISCOVER — find S1 (P2) and S2 (P1) output dirs inside the experiment tree
# ============================================================
import collections, re
EXP=f'{BASE}/gpt_image_1_5_experiment'
assert os.path.isdir(EXP), EXP
PAT=re.compile(r'(\d{4})_mid(?:_r\d+)?\.png$')
groups=collections.defaultdict(list)
for dp,_,fns in os.walk(f'{EXP}/runs'):
    for fn in fns:
        if fn.endswith('.png') and PAT.search(fn): groups[dp].append(fn)
for dp in sorted(groups):
    print(f"{len(groups[dp]):4d} files   {dp.replace(EXP+'/','')}   e.g. {sorted(groups[dp])[0]}")

In [ ]:
# ============================================================
# GPT B. STAGE — copy S1 -> tm_scale1000/gpt/{k:04d}.png, S2 -> tm_scale1000/gpt_p1/{k:04d}.png
# ============================================================
import shutil
S1_DIR=f'{EXP}/runs/<PASTE-S1-DIR>'     # P2 directed  (the paper-aligned row)
S2_DIR=f'{EXP}/runs/<PASTE-S2-DIR>'     # P1 symmetric
SUB=pd.read_csv(f'{BASE}/pairs_120/pairs.csv'); KS120=[int(k) for k in SUB.k]
for src,name in [(S1_DIR,'gpt'),(S2_DIR,'gpt_p1')]:
    dst=f'{SCALE}/{name}'; os.makedirs(dst,exist_ok=True)
    n=0
    for f in glob.glob(f'{src}/*.png'):
        m=re.search(r'(\d{4})_mid\.png$',f)          # rep-1 exports only; skips _r02 etc.
        if m: shutil.copy(f,f'{dst}/{m.group(1)}.png'); n+=1
    have={int(os.path.basename(f)[:4]) for f in glob.glob(f'{dst}/*.png')}
    miss=[k for k in KS120 if k not in have]
    print(f'{name}: copied {n}, total {len(have)}, missing {len(miss)} {miss[:5]}')
assert all(os.path.exists(f'{SCALE}/{n}/{k:04d}.png') for n in ['gpt','gpt_p1'] for k in KS120), 'missing files — stop'

In [ ]:
for run in ['full_pending_approval_20260901','preview_all_series_20260901']:
    for s in ['S1','S2']:
        d=f'{EXP}/runs/{run}/images/{s}'
        if os.path.isdir(d):
            fs=sorted(os.listdir(d))
            print(f'{run}/{s}: {len(fs)} entries, e.g. {fs[:3]}')
        else:
            print(f'{run}/{s}: MISSING')

In [ ]:
FULL=f'{EXP}/runs/full_pending_approval_20260901'
for dp,dns,fns in os.walk(FULL):
    pngs=[f for f in fns if f.endswith('.png')]
    rel=dp.replace(FULL,'') or '/'
    if pngs or dns: print(f'{rel:60} dirs={dns[:6]} pngs={len(pngs)} e.g. {sorted(pngs)[:2]}')

In [ ]:
FULL=f'{EXP}/runs/full_pending_approval_20260901'
import subprocess
r=subprocess.run(['bash','-c',f'find "{FULL}" -name "*.png" | wc -l; find "{FULL}" -name "*.png" | head -15'],
                 capture_output=True,text=True)
print(r.stdout or r.stderr)

In [ ]:
# ============================================================
# GPT C. SCORE — same-120-pair table + paired tests, canonical 128px protocol
# ============================================================
KSET=set(KS120); SCORE_RES=128
def _open128(f): return Image.open(f).convert('RGB').resize((SCORE_RES,SCORE_RES))
anc=dict(np.load(f'{OUT}/anchor_dino.npz'))
METHODS={m:p for m,p in {'ot':f'{SCALE}/ot','exact':f'{SCALE}/exact','ot_sym':f'{SCALE}/ot_sym',
    'gaussian_w2':f'{SCALE}/gaussian_w2','pixel_linear':f'{SCALE}/pixel_linear',
    'texton_linear':f'{SCALE}/texton_linear','tm':f'{SCALE}/mid',
    'gpt':f'{SCALE}/gpt','gpt_p1':f'{SCALE}/gpt_p1'}.items() if os.path.isdir(p)}
print('methods:',list(METHODS))

rows=[]; XIs={}
for m,pdir in METHODS.items():
    fI,fD=f'{OUT}/feat_{m}_inc.npy',f'{OUT}/feat_{m}_dino.npy'
    ks_all=[int(p_['k']) for p_ in MAN_ALL if os.path.exists(f'{pdir}/{int(p_["k"]):04d}.png')]
    if os.path.exists(fI) and len(ks_all)>=1000 and len(np.load(fI))==len(ks_all):
        XI_all,XD_all=np.load(fI),np.load(fD)                      # cached full-run feats -> index the 120
        idx=[i for i,k in enumerate(ks_all) if k in KSET]
        XI,XD,ks=XI_all[idx],XD_all[idx],[ks_all[i] for i in idx]
    else:                                                          # gpt rows: fresh at 128px, same as everything else
        fs=[f'{pdir}/{k:04d}.png' for k in KS120]
        ks=KS120[:]; ims=[_open128(f) for f in fs]; XI=inc_feats(ims); XD=dino_feats(ims)
    XIs[m]=XI
    rI=realism(XI,MANI_INC); rD=realism(XD,MANI_DINO)
    plc=np.array([np.linalg.norm(XD[i]-anc[f'{k}_A'])/(np.linalg.norm(XD[i]-anc[f'{k}_A'])+np.linalg.norm(XD[i]-anc[f'{k}_B'])+1e-9) for i,k in enumerate(ks)])
    for i,k in enumerate(ks): rows.append(dict(k=k,method=m,real_inc=rI[i],real_dino=rD[i],place=plc[i]))
    print(f'{m}: {len(ks)}/120')
sc=pd.DataFrame(rows); sc.to_csv(f'{OUT}/gpt_subset_scores.csv',index=False)

piv={c:sc.pivot(index='k',columns='method',values=c) for c in ['real_inc','real_dino','place']}
print(f"\n=== SAME 120 PAIRS, canonical 128px (FID descriptive at n=120; KID + paired tests carry the comparison) ===")
print(f"{'method':14}{'FID120':>8}{'KID':>16}{'realInc':>9}{'realDINO':>10}{'|place-.5|':>11}")
res={}
for m in METHODS:
    Fv=fid(XIs[m],ref_inc); Kv=kid(XIs[m],ref_inc,n_sub=min(120,len(XIs[m])))
    res[m]=dict(FID=round(Fv,1),KID=round(Kv[0],4),KIDsd=round(Kv[1],4),RInc=round(float(piv['real_inc'][m].median()),3),
                RDin=round(float(piv['real_dino'][m].median()),3),Place=round(float((piv['place'][m]-.5).abs().median()),3))
    print(f"{m:14}{Fv:8.1f}{Kv[0]:9.4f}±{Kv[1]:.4f}{res[m]['RInc']:9.3f}{res[m]['RDin']:10.3f}{res[m]['Place']:11.3f}")
json.dump(res,open(f'{OUT}/gpt_subset_numbers.json','w'),indent=1)

for ref in ['gpt','gpt_p1']:
    print(f"\n--- paired per-pair vs {ref} ({'S1/P2 directed' if ref=='gpt' else 'S2/P1 symmetric'}); + = {ref} higher ---")
    print(f"{'method':14}{'dRealInc':>28}{'dRealDINO':>28}{'d|place-.5|':>28}")
    for m in METHODS:
        if m==ref: continue
        line=f"{m:14}"
        for c,absang in [('real_inc',0),('real_dino',0),('place',1)]:
            a,b=piv[c][ref],piv[c][m]
            if absang: a=(a-.5).abs(); b=(b-.5).abs()
            dlt=(a-b).dropna().values; lo,hi=boot_ci(dlt)
            line+=f"  {np.median(dlt):+.3f}[{lo:+.3f},{hi:+.3f}]p={wilcoxon(dlt).pvalue:.1g}"
        print(line)

In [ ]:
# ============================================================
# GPT D. STRIPS — A | OT | GPT-P2 | GPT-P1 | TM | B
# ============================================================
D=f'{BASE}/paper_figs/gpt'; os.makedirs(D,exist_ok=True)
show=sorted(KS120)[:8]
cols=[('A',f'{SCALE}/A'),('OT',f'{SCALE}/ot'),('GPT P2',f'{SCALE}/gpt'),('GPT P1',f'{SCALE}/gpt_p1'),('TM',f'{SCALE}/mid'),('B',f'{SCALE}/B')]
fig,ax=plt.subplots(len(show),len(cols),figsize=(2.0*len(cols),2.0*len(show)))
for i,k in enumerate(show):
    for j,(t,p) in enumerate(cols):
        ax[i,j].imshow(Image.open(f'{p}/{k:04d}.png').convert('RGB').resize((256,256))); ax[i,j].axis('off')
        if i==0: ax[i,j].set_title(t,fontsize=9)
    ax[i,0].set_ylabel(f'k={k}',fontsize=7,rotation=0,ha='right',va='center',labelpad=18)
plt.tight_layout(); plt.savefig(f'{D}/strip_first8.png',dpi=180,bbox_inches='tight'); plt.show()

In [ ]:
# minimal deps for the ref-list cell: DTD locally + Inception observer
import os, glob, time, json
import numpy as np, torch, torch.nn.functional as F
from PIL import Image
device='cuda'
LOCAL='/content/dtd_images'; ZIP=f'{BASE}/dtd_images.zip'
if len(glob.glob(f'{LOCAL}/*/*.jpg'))<5000:
    os.system(f'unzip -qo "{ZIP}" -d /content/_dtd && mv /content/_dtd/images {LOCAL}')
print('DTD:',len(glob.glob(f'{LOCAL}/*/*.jpg')),'images')

from torchvision.models import inception_v3, Inception_V3_Weights
_inc=inception_v3(weights=Inception_V3_Weights.IMAGENET1K_V1,aux_logits=True).to(device).eval(); _inc.fc=torch.nn.Identity()
for p in _inc.parameters(): p.requires_grad_(False)
_M=torch.tensor([0.485,0.456,0.406],device=device).view(1,3,1,1); _S=torch.tensor([0.229,0.224,0.225],device=device).view(1,3,1,1)
def _stack(pils): return torch.stack([torch.from_numpy(np.asarray(p.convert('RGB').resize((256,256))).copy()).float().permute(2,0,1)/255. for p in pils]).to(device)
@torch.no_grad()
def inc_feats(pils,bs=64):
    out=[]
    for i in range(0,len(pils),bs):
        b=F.interpolate(_stack(pils[i:i+bs]),size=299,mode='bilinear',align_corners=False); out.append(_inc((b-_M)/_S).cpu().numpy())
    return np.concatenate(out)
print('observer ready')

In [ ]:
# ============================================================
# REGENERATE ref_list.txt from frozen ref_inc.npy by exhaustive matching.
# Run in Colab AFTER finish_day.ipynb's setup cell (needs inc_feats, LOCAL, OUT, BASE).
# ~10 min on GPU: computes Inception features for all DTD images at the
# canonical 128px chain, then matches each saved reference row to its
# unique nearest DTD image and verifies the match is exact.
# Output: {BASE}/pairs_120/ref_list.txt  (DTD-relative paths, sorted by row order of ref_inc.npy)
# ============================================================
ref_inc = np.load(f'{OUT}/ref_inc.npy')
print('reference rows:', ref_inc.shape)

all_files = sorted(glob.glob(f'{LOCAL}/*/*.jpg'))
print('DTD images:', len(all_files))

# features for ALL DTD images through the exact canonical chain
CAND = f'{OUT}/dtd_all_inc128.npy'
if os.path.exists(CAND) and len(np.load(CAND)) == len(all_files):
    Xall = np.load(CAND)
else:
    ims = [Image.open(f).convert('RGB').resize((128, 128)) for f in all_files]
    Xall = inc_feats(ims)
    np.save(CAND, Xall)
print('candidate features:', Xall.shape)

# match every reference row to its nearest DTD image (chunked to fit memory)
import numpy.linalg as npl
matches = np.empty(len(ref_inc), dtype=int); dists = np.empty(len(ref_inc))
CH = 256
Xn = (Xall ** 2).sum(1)
for i in range(0, len(ref_inc), CH):
    R = ref_inc[i:i+CH]
    D = Xn[None, :] - 2 * R @ Xall.T + (R ** 2).sum(1)[:, None]   # squared distances
    matches[i:i+CH] = D.argmin(1); dists[i:i+CH] = D.min(1)

print(f'max match distance: {dists.max():.3e}   (exact match => ~1e-4 or below, float32 noise)')
print(f'unique matched files: {len(set(matches))} / {len(ref_inc)}')
assert len(set(matches)) == len(ref_inc), 'DUPLICATE MATCH — stop and report, do not commit'
assert dists.max() < 1.0, 'MATCHES NOT EXACT — the saved features were not built with this chain; stop and report'

rel = [all_files[j].replace(f'{LOCAL}/', '') for j in matches]
with open(f'{BASE}/pairs_120/ref_list.txt', 'w') as fh:
    fh.write('\n'.join(rel) + '\n')
print('wrote', f'{BASE}/pairs_120/ref_list.txt', '-', len(rel), 'lines')
print('sample:', rel[:3])

In [ ]:
# ============================================================
# ONE CELL: mount Drive -> unzip DTD -> load Inception -> regenerate ref_list.txt
# Output: {BASE}/pairs_120/ref_list.txt  (~12-15 min total on T4)
# ============================================================
from google.colab import drive
drive.mount('/content/drive')

import os, glob, time
import numpy as np, torch, torch.nn.functional as F
from PIL import Image

assert torch.cuda.is_available(), 'runtime is not GPU — Runtime > Change runtime type > T4 GPU'
device='cuda'
BASE=os.environ.get('DATA_ROOT', '/content/data')  # was hardcoded to the Drive mount; set DATA_ROOT to point at your local copy
OUT=f'{BASE}/matching_ablation'

# --- frozen reference features ---
ref_inc=np.load(f'{OUT}/ref_inc.npy')
print('reference rows:', ref_inc.shape)

# --- DTD locally ---
LOCAL='/content/dtd_images'; ZIP=f'{BASE}/dtd_images.zip'
print('zip exists:', os.path.exists(ZIP))
if len(glob.glob(f'{LOCAL}/*/*.jpg'))<5000:
    os.system('rm -rf /content/_dtd')
    rc=os.system(f'unzip -qo "{ZIP}" -d /content/_dtd')
    inner=os.listdir('/content/_dtd') if os.path.isdir('/content/_dtd') else []
    print('unzip rc:', rc, '| extracted:', inner)
    src='/content/_dtd/images' if 'images' in inner else (f'/content/_dtd/{inner[0]}' if len(inner)==1 else None)
    assert src and os.path.isdir(src), f'unexpected zip layout: {inner} — stop and report'
    os.system(f'rm -rf {LOCAL} && mv "{src}" {LOCAL}')
all_files=sorted(glob.glob(f'{LOCAL}/*/*.jpg'))
print('DTD images:', len(all_files))
assert len(all_files)>5000, 'DTD did not extract — stop and report'

# --- Inception observer (canonical chain) ---
from torchvision.models import inception_v3, Inception_V3_Weights
_inc=inception_v3(weights=Inception_V3_Weights.IMAGENET1K_V1,aux_logits=True).to(device).eval(); _inc.fc=torch.nn.Identity()
for p in _inc.parameters(): p.requires_grad_(False)
_M=torch.tensor([0.485,0.456,0.406],device=device).view(1,3,1,1); _S=torch.tensor([0.229,0.224,0.225],device=device).view(1,3,1,1)
def _stack(pils): return torch.stack([torch.from_numpy(np.asarray(p.convert('RGB').resize((256,256))).copy()).float().permute(2,0,1)/255. for p in pils]).to(device)
@torch.no_grad()
def inc_feats(pils,bs=64):
    out=[]
    for i in range(0,len(pils),bs):
        b=F.interpolate(_stack(pils[i:i+bs]),size=299,mode='bilinear',align_corners=False); out.append(_inc((b-_M)/_S).cpu().numpy())
    return np.concatenate(out)
print('observer ready')

# --- features for all DTD images through the canonical 128px chain (cached on Drive) ---
CAND=f'{OUT}/dtd_all_inc128.npy'
if os.path.exists(CAND) and len(np.load(CAND,mmap_mode='r'))==len(all_files):
    Xall=np.load(CAND); print('loaded cached candidate features')
else:
    Xall=np.empty((len(all_files),ref_inc.shape[1]),dtype=np.float32); t0=time.time()
    B=256
    for i in range(0,len(all_files),B):
        ims=[Image.open(f).convert('RGB').resize((128,128)) for f in all_files[i:i+B]]
        Xall[i:i+len(ims)]=inc_feats(ims)
        print(f'{min(i+B,len(all_files))}/{len(all_files)}  {time.time()-t0:.0f}s',end='\r')
    np.save(CAND,Xall); print('\ncandidate features done')

# --- match each reference row to its unique nearest DTD image ---
matches=np.empty(len(ref_inc),dtype=int); dists=np.empty(len(ref_inc))
Xn=(Xall.astype(np.float64)**2).sum(1)
for i in range(0,len(ref_inc),256):
    R=ref_inc[i:i+256].astype(np.float64)
    D=Xn[None,:]-2*R@Xall.T.astype(np.float64)+(R**2).sum(1)[:,None]
    matches[i:i+256]=D.argmin(1); dists[i:i+256]=D.min(1)
print(f'max match distance: {dists.max():.3e}  (float noise expected; large values = mismatch)')
print(f'unique matched files: {len(set(matches))}/{len(ref_inc)}')
assert len(set(matches))==len(ref_inc), 'DUPLICATE MATCH — do not commit; paste this output back'
assert dists.max()<1.0, 'MATCHES NOT EXACT — do not commit; paste this output back'

# --- write the list ---
rel=[all_files[j].replace(f'{LOCAL}/','') for j in matches]
os.makedirs(f'{BASE}/pairs_120',exist_ok=True)
with open(f'{BASE}/pairs_120/ref_list.txt','w') as fh: fh.write('\n'.join(rel)+'\n')
print('WROTE', f'{BASE}/pairs_120/ref_list.txt', '-', len(rel), 'lines')
print('sample:', rel[:3])

In [ ]:
# ============================================================
# FIX: resolve the 4 collisions — check they're identical-image ties, reassign to distinct twins
# ============================================================
import collections
dup_targets=[j for j,c in collections.Counter(matches).items() if c>1]
print('collided DTD files:')
for j in dup_targets: print('  ',all_files[j].replace(f'{LOCAL}/',''))

# rows involved + their top-5 nearest candidates
rows=[i for i in range(len(matches)) if matches[i] in dup_targets]
print('\nrows involved:',rows)
for i in rows:
    R=ref_inc[i].astype(np.float64)
    D=Xn-2*R@Xall.T.astype(np.float64)+(R**2).sum()
    top=np.argsort(D)[:5]
    print(f'row {i}: '+'  '.join(f'{all_files[j].split("/")[-1]}:{D[j]:.2e}' for j in top))

# greedy reassignment: give each colliding row a distinct candidate among its near-ties
TIE_TOL=1.0
taken=set(matches[i] for i in range(len(matches)) if i not in rows)
new=matches.copy()
for i in rows:
    R=ref_inc[i].astype(np.float64)
    D=Xn-2*R@Xall.T.astype(np.float64)+(R**2).sum()
    for j in np.argsort(D):
        if D[j]>=TIE_TOL: raise RuntimeError(f'row {i}: no unused near-tie candidate — paste output back')
        if j not in taken: new[i]=j; taken.add(j); break
matches=new
uniq=len(set(matches)); mx=max(float((Xn[matches[i]]-2*ref_inc[i].astype(np.float64)@Xall[matches[i]].astype(np.float64)+ (ref_inc[i].astype(np.float64)**2).sum())) for i in rows)
print(f'\nafter repair: unique {uniq}/{len(matches)}, max reassigned distance {mx:.3e}')
assert uniq==len(matches) and mx<1.0

rel=[all_files[j].replace(f'{LOCAL}/','') for j in matches]
with open(f'{BASE}/pairs_120/ref_list.txt','w') as fh: fh.write('\n'.join(rel)+'\n')
print('WROTE',f'{BASE}/pairs_120/ref_list.txt','-',len(rel),'lines')